# Figure S9

Draws regional groundwater reconstruction validation results.


In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import rasterio
from shapely.geometry import box

warnings.filterwarnings('ignore', category=RuntimeWarning)


def find_repo_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return p
    raise RuntimeError('Could not locate repository root.')


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS9'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = RECON / 'reconstruction' / 'wtd_reconstructed_matrix.npy'
GRID_PATH = RECON / 'metadata' / 'grid_lookup.csv'
MONTH_PATH = RECON / 'metadata' / 'month_index.csv'
REGION_PATH = ROOT / 'data' / '12 MAP generalized regions' / 'MAP_generalized_regions_mrva_clip_epsg5070.gpkg'
GRACE_CELL_PATH = ROOT / 'data' / '7 GRACE' / 'grace_mrva_2011_2023_cell_monthly.csv'
GRACE_TOTAL_PATH = RECON / 'diagnostics' / 'regional_mass_correction' / 'regional_mass_correction_monthly.csv'
CONNECTIVITY_PATH = ROOT / 'data' / '8 connectivity' / 'ConfiningLayer_SurfaceConnectivity.tif'
FIG_A_PATH = OUT_DIR / 'FigS9a_regions.png'
FIG_B_PATH = OUT_DIR / 'FigS9b_GRACE_reconstruction_validation.png'

REGION_ORDER = ['Boeuf', 'Cache-Grand Prairie', 'Delta', 'St. Francis']
REGION_MERGE = {'Cache': 'Cache-Grand Prairie', 'Grand Prairie': 'Cache-Grand Prairie'}
REGION_COLORS = {
    'Boeuf': '#4C6A92',
    'Cache-Grand Prairie': '#6FA08B',
    'Delta': '#D2A85E',
    'St. Francis': '#8E7BB5',
}
PLOT_ORDER = ['MRVA total'] + REGION_ORDER
PLOT_COLORS = {'MRVA total': '#3b3b3b', **REGION_COLORS}
SY_CLASS_MAP = {-3: 0.08, -2: 0.11, -1: 0.14, 1: 0.18, 2: 0.22, 3: 0.26}

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 9,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.major.size': 3.5,
    'ytick.major.size': 3.5,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'axes.unicode_minus': False,
    'savefig.dpi': 600,
})


def display_path(path: Path) -> str:
    return str(path.resolve().relative_to(ROOT))


In [ ]:
mat = np.load(MATRIX_PATH, mmap_mode='r')
grid = pd.read_csv(GRID_PATH)
months = pd.read_csv(MONTH_PATH)
months['month_label'] = months['month_label'].astype(str)
months['date'] = pd.to_datetime(months['month_label'] + '-01')

source_regions = gpd.read_file(REGION_PATH).to_crs('EPSG:5070')
source_regions['region'] = source_regions['region'].replace(REGION_MERGE)
regions = source_regions.dissolve(by='region', as_index=False)

grid_points = gpd.GeoDataFrame(
    grid[['grid_id', 'x', 'y']].copy(),
    geometry=gpd.points_from_xy(grid['x'], grid['y']),
    crs='EPSG:5070',
)
assignment = gpd.sjoin(
    grid_points, source_regions[['region', 'geometry']], how='left', predicate='within'
).drop(columns='index_right')
grid = grid.merge(assignment[['grid_id', 'region']], on='grid_id', how='left')
grid['matrix_col'] = np.arange(len(grid), dtype=int)

coords = list(zip(grid['x'], grid['y']))
with rasterio.open(CONNECTIVITY_PATH) as src:
    connectivity = np.array([v[0] for v in src.sample(coords)], dtype=np.float32)
    if src.nodata is not None:
        connectivity[connectivity == src.nodata] = np.nan
grid['specific_yield'] = 0.18
for cls, value in SY_CLASS_MAP.items():
    grid.loc[connectivity == cls, 'specific_yield'] = value

assigned_ll = assignment.dropna(subset=['region']).to_crs('EPSG:4326').copy()
assigned_ll['lat_idx'] = np.floor((assigned_ll.geometry.y + 90.0) / 0.25).astype(int)
assigned_ll['lon_idx'] = np.floor((assigned_ll.geometry.x % 360.0) / 0.25).astype(int)
grace_weights = (
    assigned_ll.groupby(['region', 'lat_idx', 'lon_idx'])
    .size().rename('n_grid_cells').reset_index()
)
grace_weights['cell_key'] = grace_weights['lat_idx'].astype(str) + '_' + grace_weights['lon_idx'].astype(str)
grace_cells = pd.read_csv(GRACE_CELL_PATH)


def weighted_mean(group: pd.DataFrame) -> float:
    valid = group['grace_cm'].notna() & group['n_grid_cells'].notna()
    return float(np.average(group.loc[valid, 'grace_cm'], weights=group.loc[valid, 'n_grid_cells'])) if valid.any() else np.nan


def storage_proxy(region: str, valid_grace: np.ndarray) -> np.ndarray:
    idx = grid.loc[grid['region'].eq(region), 'matrix_col'].to_numpy(dtype=int)
    region_wtd = np.asarray(mat[:, idx], dtype=np.float32)
    reference = np.nanmean(region_wtd[np.where(valid_grace)[0]], axis=0)
    sy = grid.loc[idx, 'specific_yield'].to_numpy(dtype=np.float32)
    return -100.0 * np.nanmean((region_wtd - reference) * sy[None, :], axis=1)


regional_rows = []
regional_r = {}
for region in REGION_ORDER:
    weights = grace_weights.loc[grace_weights['region'].eq(region), ['cell_key', 'n_grid_cells']]
    grace = grace_cells.merge(weights, on='cell_key', how='inner')
    grace_series = (
        grace.groupby('month_label').apply(weighted_mean, include_groups=False)
        .rename('grace_lwe_cm').reset_index()
    )
    full = months[['month_label', 'date']].merge(grace_series, on='month_label', how='left')
    valid = full['grace_lwe_cm'].notna().to_numpy()
    full['grace_lwe_anom_cm'] = full['grace_lwe_cm'] - full.loc[valid, 'grace_lwe_cm'].mean()
    full['reconstruction_storage_proxy_cm'] = storage_proxy(region, valid)
    full['region'] = region
    mask = full[['grace_lwe_anom_cm', 'reconstruction_storage_proxy_cm']].notna().all(axis=1)
    regional_r[region] = full.loc[mask, 'grace_lwe_anom_cm'].corr(full.loc[mask, 'reconstruction_storage_proxy_cm'])
    regional_rows.append(full)

total = pd.read_csv(GRACE_TOTAL_PATH)
total['month_label'] = total['month_label'].astype(str)
total = months[['month_label', 'date']].merge(total, on='month_label', how='left').rename(columns={
    'grace_anom_cm': 'grace_lwe_anom_cm',
    'corrected_storage_proxy_cm': 'reconstruction_storage_proxy_cm',
})
total['region'] = 'MRVA total'
mask = total[['grace_lwe_anom_cm', 'reconstruction_storage_proxy_cm']].notna().all(axis=1)
regional_r['MRVA total'] = total.loc[mask, 'grace_lwe_anom_cm'].corr(total.loc[mask, 'reconstruction_storage_proxy_cm'])
regional_ts = pd.concat([total, *regional_rows], ignore_index=True)

grace_unique = grace_weights[['lat_idx', 'lon_idx', 'cell_key']].drop_duplicates().copy()
grace_unique['lon_min'] = grace_unique['lon_idx'] * 0.25
grace_unique['lon_max'] = (grace_unique['lon_idx'] + 1) * 0.25
grace_unique['lat_min'] = grace_unique['lat_idx'] * 0.25 - 90.0
grace_unique['lat_max'] = (grace_unique['lat_idx'] + 1) * 0.25 - 90.0
for col in ['lon_min', 'lon_max']:
    grace_unique[col] = grace_unique[col].where(grace_unique[col] <= 180, grace_unique[col] - 360)
grace_boxes = gpd.GeoDataFrame(
    grace_unique,
    geometry=[box(r.lon_min, r.lat_min, r.lon_max, r.lat_max) for r in grace_unique.itertuples()],
    crs='EPSG:4326',
)
regions_ll = regions.to_crs('EPSG:4326')


In [ ]:
FIG_HEIGHT_IN = 5.8
region_handles = [Patch(facecolor=REGION_COLORS[r], edgecolor='none', label=r) for r in REGION_ORDER]
region_handles.append(Patch(facecolor='none', edgecolor='#9c9c9c', linewidth=0.7, label='GRACE grid'))

fig_a, ax_a = plt.subplots(figsize=(3.35, FIG_HEIGHT_IN), dpi=600)
fig_a.subplots_adjust(left=0.14, right=0.98, bottom=0.10, top=0.98)
grace_boxes.boundary.plot(ax=ax_a, color='#9c9c9c', linewidth=0.38, alpha=0.62, zorder=1)
for region in REGION_ORDER:
    regions_ll.loc[regions_ll['region'].eq(region)].plot(
        ax=ax_a, facecolor=REGION_COLORS[region], edgecolor='white',
        linewidth=0.55, alpha=0.96, zorder=2,
    )
regions_ll.boundary.plot(ax=ax_a, color='#262626', linewidth=0.55, zorder=3)
ax_a.set_xlabel('')
ax_a.set_ylabel('')
ax_a.set_aspect(1.0 / np.cos(np.deg2rad(35.0)))
ax_a.margins(x=0.025, y=0.020)
ax_a.legend(
    handles=region_handles, loc='lower right', bbox_to_anchor=(0.98, 0.025),
    frameon=False, handlelength=1.05, handletextpad=0.45,
    labelspacing=0.32, borderaxespad=0, fontsize=7.2,
)
ax_a.tick_params(axis='both', which='both', bottom=False, left=False,
                 labelbottom=False, labelleft=False, top=False, right=False)
for spine in ax_a.spines.values():
    spine.set_visible(False)
fig_a.savefig(FIG_A_PATH, dpi=600, facecolor='white')
plt.show()

left_limit = max(25.0, np.ceil(regional_ts['grace_lwe_anom_cm'].abs().max() / 5.0) * 5.0)
right_limit = max(20.0, np.ceil(regional_ts['reconstruction_storage_proxy_cm'].abs().max() / 5.0) * 5.0)
grace_color, recon_color = '#4F91A8', '#74553B'
fig_b, axes = plt.subplots(len(PLOT_ORDER), 1, figsize=(7.6, FIG_HEIGHT_IN), sharex=True)
fig_b.subplots_adjust(left=0.115, right=0.885, bottom=0.085, top=0.935, hspace=0.14)
right_axes, legend_handles = [], None

for row, (ax, region) in enumerate(zip(axes, PLOT_ORDER)):
    sub = regional_ts.loc[regional_ts['region'].eq(region)].sort_values('date')
    ax2 = ax.twinx()
    right_axes.append(ax2)
    grace = ax.scatter(
        sub['date'], sub['grace_lwe_anom_cm'], s=18, facecolor='white',
        edgecolor=grace_color, linewidth=0.90, zorder=3, label='GRACE',
    )
    recon, = ax2.plot(
        sub['date'], sub['reconstruction_storage_proxy_cm'], color=recon_color,
        linewidth=1.55, zorder=2, label='Reconstruction',
    )
    legend_handles = legend_handles or [grace, recon]
    ax.axhline(0, color='#666666', linewidth=0.70, zorder=1)
    ax.set_ylim(-left_limit, left_limit)
    row_right_limit = 20.0 if row < 3 else right_limit
    ax2.set_ylim(-row_right_limit, row_right_limit)
    if row < 3:
        ax2.set_yticks([-20, -10, 0, 10, 20])
    ax.tick_params(axis='y', colors=grace_color, pad=3)
    ax2.tick_params(axis='y', colors=recon_color, pad=3)
    label_box = dict(facecolor='none', edgecolor='none', alpha=0.0, pad=0.0)
    ax2.text(
        0.025, 0.88, region, transform=ax2.transAxes, ha='left', va='top',
        fontsize=9.5, color=PLOT_COLORS[region], fontweight='bold', bbox=label_box, zorder=5,
    )
    ax2.text(
        0.995, 0.88, rf"$r$ = {regional_r[region]:.2f}", transform=ax2.transAxes,
        ha='right', va='top', fontsize=8.5, color='#3d3d3d', bbox=label_box, zorder=5,
    )
    ax.set_axisbelow(True)
    ax.grid(axis='x', color='#dedede', linewidth=0.45, alpha=0.65)
    ax.grid(axis='y', color='#e7e7e7', linewidth=0.40, alpha=0.70)
    ax.tick_params(top=True, right=False, direction='out')
    ax2.tick_params(top=False, left=False, right=True, direction='out')
    for spine in ax.spines.values():
        spine.set(linewidth=0.8, color='#303030')
    for side, spine in ax2.spines.items():
        spine.set_visible(side == 'right')
        if side == 'right':
            spine.set(linewidth=0.8, color='#303030')
    if row < len(PLOT_ORDER) - 1:
        ax.tick_params(labelbottom=False)

middle = len(PLOT_ORDER) // 2
axes[middle].set_ylabel('GRACE anomaly (cm)', color=grace_color, labelpad=8)
right_axes[middle].set_ylabel('Reconstruction anomaly (cm)', color=recon_color, labelpad=8)
fig_b.legend(
    legend_handles, ['GRACE', 'Reconstruction'], loc='upper center',
    bbox_to_anchor=(0.52, 0.982), ncol=2, frameon=False,
    handlelength=2.3, columnspacing=2.0, fontsize=9.2,
)
axes[-1].set_xlabel('Year')
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig_b.savefig(FIG_B_PATH, dpi=600, facecolor='white')
plt.show()

print('Saved:')
print('  ' + display_path(FIG_A_PATH))
print('  ' + display_path(FIG_B_PATH))
